# 메이플스토리 260 허들 분석 — analysis.ipynb

계획문서 §4 핵심질문: **레벨 260까지 키운 캐릭터를 계속 키우는가 방치하는가, 여름 성장 이벤트는 이를 어떻게 바꾸는가.**
데이터: NEXON Open API, 앵커 2026-06-18 ~ +34일(일간), 4코호트 (`approach` Lv.251 / `at260` Lv.260 / `past260` Lv.262 / `burnend` Lv.281).
실행 순서: `collect.py` → **이 노트북** → `REPORT.md`.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 160)

DATA = Path("data"); IMG = Path("images"); IMG.mkdir(exist_ok=True)
ANCHOR = pd.Timestamp("2026-06-18")
COHORTS = ["approach", "at260", "past260", "burnend"]
COLOR = {"approach": "#4C72B0", "at260": "#C44E52", "past260": "#55A868", "burnend": "#8172B2"}
CAPTION = "집계 단위 = 앵커일(2026-06-18) 기준 경과일(일간), 창=35일"
TARGET  = {"approach": 251, "at260": 260, "past260": 262, "burnend": 281}
EXPECT_N = {"approach": 300, "at260": 800, "past260": 300, "burnend": 300}
STAT_OFFSETS = [0, 8, 16, 24, 32]
DTYPE = {"cohort": "string", "character_name": "string", "ocid": "string",
         "level": "Int64", "exp": "Int64", "exp_rate": "float64",
         "combat_power": "Int64", "error_code": "string"}

df = pd.concat(
    [pd.read_csv(DATA / f"cohort_{c}.csv", dtype=DTYPE, parse_dates=["date"]) for c in COHORTS],
    ignore_index=True,
)
df["elapsed_day"] = (df["date"] - ANCHOR).dt.days
df["cohort"] = pd.Categorical(df["cohort"], categories=COHORTS, ordered=True)
print(df.shape)
df.head()

(59500, 10)


,cohort,character_name,ocid,date,level,exp,exp_rate,combat_power,error_code,elapsed_day
0,approach,노현솔,db3cade0511875435786133c6e2304b2,2026-06-18,251,247760164,0.129,0,<NA>,0
1,approach,노현솔,db3cade0511875435786133c6e2304b2,2026-06-19,251,247760164,0.129,<NA>,<NA>,1
2,approach,노현솔,db3cade0511875435786133c6e2304b2,2026-06-20,251,247760164,0.129,<NA>,<NA>,2
3,approach,노현솔,db3cade0511875435786133c6e2304b2,2026-06-21,251,247760164,0.129,<NA>,<NA>,3
4,approach,노현솔,db3cade0511875435786133c6e2304b2,2026-06-22,251,247760164,0.129,<NA>,<NA>,4


In [2]:
# 미션 §4 — 기간 / 컬럼·dtype / 코호트별 행수 / 결측 / 중복 / 날짜 연속성 / 앵커 레벨
print("기간 :", df.date.min().date(), "~", df.date.max().date(),
      "| elapsed_day", int(df.elapsed_day.min()), "~", int(df.elapsed_day.max()))
print("\n[dtype]"); print(df.dtypes)

g = df.groupby("cohort", observed=True)
info = pd.DataFrame({
    "행수": g.size(),
    "기대행수": pd.Series({c: EXPECT_N[c] * 35 for c in COHORTS}),
    "캐릭터수": g.character_name.nunique(),
    "날짜수": g.date.nunique(),
})
print("\n[코호트별 규모]"); print(info)

print("\n[결측 df.isna().sum()]"); print(df.isna().sum())

dup = df.duplicated(["cohort", "character_name", "date"]).sum()
per = df.groupby(["cohort", "character_name"], observed=True).elapsed_day.agg(["min", "max", "count"])
print(f"\n(cohort,character_name,date) 중복 행: {dup}")
print(f"캐릭터별 관측일수 35 아닌 캐릭터: {(per['count'] != 35).sum()} "
      f"(min={per['count'].min()}, max={per['count'].max()})")
print(f"elapsed_day 범위: 전 캐릭터 min={per['min'].min()}, max={per['max'].max()}  (0~34 연속)")

d0 = df[df.elapsed_day == 0]
print("\n[앵커일(D+0) 레벨 vs 코호트 목표]")
for c in COHORTS:
    s = d0[(d0.cohort == c) & d0.level.notna()]
    print(f"  {c:9s}: {s.level.min()}~{s.level.max()}  (목표 {TARGET[c]}, |차이|>5 인 캐릭터 {(s.level - TARGET[c]).abs().gt(5).sum()}명)")

기간 : 2026-06-18 ~ 2026-07-22 | elapsed_day 0 ~ 34

[dtype]
cohort                  category
character_name    string[python]
ocid              string[python]
date              datetime64[ns]
level                      Int64
exp                        Int64
exp_rate                 float64
combat_power               Int64
error_code        string[python]
elapsed_day                int64
dtype: object

[코호트별 규모]
             행수   기대행수  캐릭터수  날짜수
approach  10500  10500   300   35
at260     28000  28000   800   35
past260   10500  10500   300   35
burnend   10500  10500   300   35

[결측 df.isna().sum()]
cohort                0
character_name        0
ocid               1470
date                  0
level              2086
exp                2086
exp_rate           2086
combat_power      51299
error_code        57491
elapsed_day           0
dtype: int64

(cohort,character_name,date) 중복 행: 0
캐릭터별 관측일수 35 아닌 캐릭터: 0 (min=35, max=35)
elapsed_day 범위: 전 캐릭터 min=0, max=34  (0~34 연속)

[앵커일(D+0) 레벨 vs

In [3]:
# 스펙 §2.1 — 조회 실패는 "무활동(0)" 아니라 별도 이탈 신호. 비율 분모에서 제외하고 n_dropout 으로 집계.
# http_400 : ocid 는 유효하나 /character/basic 이 전 기간 400 → 조회불가 계정, http_404 와 동일 취급
#            (스펙 §2.1 표에 없던 코드 — 실수집에서 15명/525행 발견, REPORT §4 에 명시)
DROP_CODES = {"ocid_failed", "http_404", "http_400",
              "http_429_giveup", "http_5xx_giveup", "timeout"}
df["is_dropout"] = df.error_code.isin(DROP_CODES)

char_drop = df.groupby(["cohort", "character_name"], observed=True).is_dropout.mean()
full_drop = char_drop[char_drop == 1]
print("[코호트별 완전 이탈 캐릭터 (35일 전부 조회 실패)]")
print(full_drop.reset_index().groupby("cohort", observed=True).size(), "\n")

err_break = (df[df.is_dropout].groupby(["cohort", "error_code"], observed=True)
             .character_name.nunique().rename("캐릭터수"))
print("[이탈 사유별 캐릭터 수]"); print(err_break, "\n")

n_dropout = (df[df.is_dropout].groupby(["cohort", "elapsed_day"], observed=True)
             .character_name.nunique().rename("n_dropout"))

clean = df[~df.is_dropout].copy().sort_values(["cohort", "character_name", "date"])

# --- 활동성 base-rate : 35일간 레벨/exp/전투력 중 하나라도 변동한 캐릭터 비율 ---
def _activity(sub):
    return pd.Series({
        "레벨변동": bool(sub.level.diff().fillna(0).ne(0).any()),
        "exp변동": bool(sub.exp.diff().fillna(0).ne(0).any()),
        "전투력변동": bool(sub.combat_power.dropna().nunique() > 1),
    })

act = (clean.groupby(["cohort", "character_name"], observed=True)[["level", "exp", "combat_power"]]
       .apply(_activity))
act["활동"] = act.any(axis=1)
base = act.groupby("cohort", observed=True).mean().round(3)
base["유효표본"] = act.groupby("cohort", observed=True).size()
print("[활동성 base-rate — 변동 캐릭터 비율]")
print(base)

[코호트별 완전 이탈 캐릭터 (35일 전부 조회 실패)]
cohort
approach    11
at260       26
past260     16
burnend      4
dtype: int64 

[이탈 사유별 캐릭터 수]
cohort    error_code 
approach  http_400        3
          ocid_failed     8
at260     http_400        6
          ocid_failed    20
past260   http_400        6
          ocid_failed    10
burnend   ocid_failed     4
Name: 캐릭터수, dtype: int64 



[활동성 base-rate — 변동 캐릭터 비율]
           레벨변동  exp변동  전투력변동     활동  유효표본
cohort                                    
approach  0.003  0.021  0.090  0.090   289
at260     0.004  0.034  0.119  0.120   774
past260   0.004  0.109  0.215  0.218   284
burnend   0.098  0.470  0.632  0.639   296


## 시계열 특성 (미션 §3 — 트렌드 / 계절성 / 노이즈)

- **트렌드**: 코호트별 평균 레벨·돌파율의 시간축 변화. 아래 셀에서 확인하듯 `approach`/`at260`/`past260` 는
  35일간 평균 레벨이 **사실상 수평**(변동 캐릭터 <15%), `burnend` 만 완만한 우상향. 즉 대부분 코호트에서
  "성장 트렌드"라 부를 신호가 거의 없다는 것 자체가 핵심 관찰.
- **계절성**: 관측 35일(=5주). 요일별 평균 활동률을 점검하면(아래 셀) 주말이 평일보다 소폭 높은
  약한 주간 패턴이 보이나, 전 코호트 활동률 자체가 6% 미만이라 실질 영향이 없음 → **계절성 기각**.
- **노이즈**: 개별 캐릭터 궤적의 소수 진동(전투력 리롤, 조회 일시 실패)과 표본 크기(코호트당 289~774명).
  → 코호트 곡선은 **이동평균(window=3)** 으로 평활, 비율 지표는 **부트스트랩 95% CI** 를 함께 제시.

In [4]:
grp = clean.groupby(["cohort", "character_name"], observed=True)

# §5.1 260 돌파 (누적) / 261 이탈 (at260 플래토 탈출)
clean["brk260"]     = (clean.level >= 260).fillna(False)
clean["brk260_cum"] = grp["brk260"].cummax()
clean["brk261"]     = (clean.level >= 261).fillna(False)
clean["brk261_cum"] = grp["brk261"].cummax()

# §5.8 레벨 증가율 (레벨/일)
clean["growth_rate"] = grp["level"].diff()

# §5.5 활동_t (전일 대비) : 레벨 변화 OR 전투력(관측일) 직전관측 대비 변화
clean["cp_ffill"]  = grp["combat_power"].ffill()
clean["lvl_moved"] = grp["level"].diff().fillna(0).ne(0)
clean["cp_moved"]  = grp["cp_ffill"].diff().fillna(0).ne(0)
clean["active_t"]  = clean.lvl_moved | clean.cp_moved

# §2.1 이상치
clean["anomaly_level_drop"] = grp["level"].diff().lt(0)        # 레벨 감소 = 물리적 불가 → 결측 취급
clean["cp_drop"] = grp["cp_ffill"].pct_change(fill_method=None).lt(-0.30)   # 전투력 -30%↓ : 이상치 아님, note 만
n_drop_anom = int(clean.anomaly_level_drop.sum())
if n_drop_anom:
    clean.loc[clean.anomaly_level_drop, ["level", "brk260", "brk260_cum", "brk261", "brk261_cum"]] = pd.NA
print(f"anomaly_level_drop (레벨 감소 스냅샷): {n_drop_anom}행 → 결측 처리")
print(f"cp_drop (전투력 -30%↓, note only): {int(clean.cp_drop.sum())}행")

# §5.3 버닝추정 (탐색적) — approach·at260 만. at260(D+0=260)은 '260 도달' 절이 자명 → '큰 점프' 절만 유효
def _burn_suspect(sub):
    sub = sub.sort_values("date")
    d0 = sub.level.iloc[0]
    if pd.isna(d0):
        return False
    early_max = sub.loc[sub.elapsed_day <= 7, "level"].max()
    crossed_fast = (d0 < 260) and pd.notna(early_max) and (early_max >= 260)
    big_jump = pd.notna(sub.level.max()) and (sub.level.max() - d0 >= 15)
    return bool(crossed_fast or big_jump)

bs = (clean[clean.cohort.isin(["approach", "at260"])]
      .groupby(["cohort", "character_name"], observed=True)
      .apply(_burn_suspect, include_groups=False).rename("burn_suspect"))
clean = clean.merge(bs, on=["cohort", "character_name"], how="left")
clean["burn_suspect"] = clean["burn_suspect"].astype("boolean").fillna(False).astype(bool)
print("\n[버닝추정 캐릭터 수]")
print(clean[clean.cohort.isin(["approach", "at260"])]
      .groupby("cohort", observed=True).apply(lambda s: s.groupby("character_name").burn_suspect.first().sum(),
                                              include_groups=False))

anomaly_level_drop (레벨 감소 스냅샷): 0행 → 결측 처리
cp_drop (전투력 -30%↓, note only): 52행



[버닝추정 캐릭터 수]
cohort
approach    0
at260       0
dtype: int64


In [5]:
def boot_ci(vals, n_resamples=10000, seed=42):
    # 비율(0/1 배열)의 부트스트랩 95% CI. 표본<2 또는 전부 동일하면 점추정 반환.
    v = np.asarray(vals, dtype=float)
    v = v[~np.isnan(v)]
    if v.size < 2 or np.all(v == v[0]):
        m = float(v.mean()) if v.size else np.nan
        return m, m
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r = st.bootstrap((v,), np.mean, confidence_level=0.95, n_resamples=n_resamples,
                         method="percentile", random_state=np.random.default_rng(seed))
    return float(r.confidence_interval.low), float(r.confidence_interval.high)

BRK_COL = {"approach": "brk260_cum", "at260": "brk261_cum"}   # past260·burnend : 돌파율 대상 아님

rows = []
for c in COHORTS:
    sub = clean[clean.cohort == c]
    for d, day_df in sub.groupby("elapsed_day", observed=True):
        rec = {"cohort": c, "date": (ANCHOR + pd.Timedelta(days=int(d))).date().isoformat(),
               "elapsed_day": int(d),
               "n": int(day_df.level.notna().sum()),
               "n_dropout": int(n_dropout.get((c, d), 0))}
        if c in BRK_COL:
            bvals = day_df[BRK_COL[c]].astype("float")
            rec["breakthrough_rate"] = float(bvals.mean())
            rec["bt_ci_lo"], rec["bt_ci_hi"] = boot_ci(bvals)
        else:
            rec["breakthrough_rate"] = rec["bt_ci_lo"] = rec["bt_ci_hi"] = np.nan
        if d >= 1:
            avals = day_df["active_t"].astype("float")
            rec["retention_rate"] = float(avals.mean())
            rec["ret_ci_lo"], rec["ret_ci_hi"] = boot_ci(avals)
        else:
            rec["retention_rate"] = rec["ret_ci_lo"] = rec["ret_ci_hi"] = np.nan
        rows.append(rec)

summary = pd.DataFrame(rows)

# --- 정착 (§5.4) : 돌파 후 남은 관측 중 최소 1회 레벨/전투력 변화 ---
# 돌파 기준 컬럼: approach=260 도달, at260=261 이탈(플래토 탈출).
# past260·burnend 는 이미 260 초과 상태 → "관측 전 구간에서 계속 성장/활동하는가"로 해석(돌파시점=D+0).
SETTLE_COL = {"approach": "brk260_cum", "at260": "brk261_cum",
              "past260": None, "burnend": None}

def settle_flag(sub, col):
    sub = sub.sort_values("date")
    if col is None:                                # 이미 260 초과 : 전 구간 활동 여부
        after = sub.iloc[1:]
        return int(after.lvl_moved.any() or after.cp_moved.any())
    hit = sub[col].astype("boolean").fillna(False)
    if not hit.any():
        return np.nan
    after = sub[hit].iloc[1:]                       # 돌파 시점 이후
    if after.empty:
        return np.nan                              # 마지막 날 첫 돌파 → 남은 관측 없음
    return int(after.lvl_moved.any() or after.cp_moved.any())

settle_rows = []
for c in COHORTS:
    sub = clean[clean.cohort == c]
    s = sub.groupby("character_name", observed=True).apply(
        settle_flag, col=SETTLE_COL[c], include_groups=False)
    k = int(s.notna().sum())                       # 돌파자 수 (남은 관측 있는)
    j = int(s.fillna(0).sum())                     # 그중 정착
    settle_rows.append({"cohort": c, "돌파자(정착판정가능)": k, "정착": j,
                        "settle_rate": (j / k) if k >= 5 else np.nan})
settle_df = pd.DataFrame(settle_rows)
summary = summary.merge(settle_df[["cohort", "settle_rate"]], on="cohort", how="left")

summary.to_csv(DATA / "breakthrough_retention_summary.csv", index=False, encoding="utf-8-sig")
print("→ data/breakthrough_retention_summary.csv", summary.shape)

print("\n[Q1 돌파 — 최종일(D+34) 기준]")
last = summary[summary.elapsed_day == 34]
for c in BRK_COL:
    r = last[last.cohort == c].iloc[0]
    lab = "260 도달률" if c == "approach" else "261 이탈률(플래토 탈출)"
    print(f"  {c:9s} {lab}: {r.breakthrough_rate:.4f}  "
          f"[95% CI {r.bt_ci_lo:.4f}, {r.bt_ci_hi:.4f}]  (n={int(r.n)})")

print("\n[정착 (§5.4) — 분모 작아 원자료로 표기]")
print(settle_df.to_string(index=False))

# --- Fisher exact : approach 버닝추정 vs 비버닝 (D+34 260 돌파) ---
ap = clean[(clean.cohort == "approach") & (clean.elapsed_day == 34)]
ct = pd.crosstab(ap.burn_suspect, ap.brk260_cum.astype("boolean").fillna(False))
print("\n[Fisher exact — approach 버닝추정 × D+34 260돌파]")
print(ct)
if ct.shape == (2, 2) and ct.values.min() >= 0 and (ct.sum(axis=1) > 0).all() and ct.index.nunique() == 2:
    odds, p = st.fisher_exact(ct.values)
    print(f"  odds ratio={odds:.3f}, p={p:.4f}")
else:
    n_bs = int(ap.burn_suspect.sum())
    print(f"  계산 불가 — approach 버닝추정군 n={n_bs} (2×2 성립 안 함). "
          f"버닝추정군이 사실상 없어 '버닝추정 vs 비버닝 정착률 비교'(계획문서 세부질문 1)는 수행 불가.")

→ data/breakthrough_retention_summary.csv (140, 12)

[Q1 돌파 — 최종일(D+34) 기준]
  approach  260 도달률: 0.0035  [95% CI 0.0000, 0.0104]  (n=289)
  at260     261 이탈률(플래토 탈출): 0.0039  [95% CI 0.0000, 0.0090]  (n=774)

[정착 (§5.4) — 분모 작아 원자료로 표기]
  cohort  돌파자(정착판정가능)  정착  settle_rate
approach            1   0          NaN
   at260            3   2          NaN
 past260          284  61     0.214789
 burnend          296 187     0.631757

[Fisher exact — approach 버닝추정 × D+34 260돌파]
brk260_cum    False  True 
burn_suspect              
False           288      1
  계산 불가 — approach 버닝추정군 n=0 (2×2 성립 안 함). 버닝추정군이 사실상 없어 '버닝추정 vs 비버닝 정착률 비교'(계획문서 세부질문 1)는 수행 불가.


In [6]:
# approach(Lv.251 고정) 는 '앵커일에 251 에 정지해 있던' 인구라 등반자를 구조적으로 배제한다.
# climb 코호트: 앵커일 Lv.255~258 이면서 앵커 직전 14일간 레벨이 오른(모멘텀) 캐릭터 300명.
#   "260 을 향해 실제로 오르고 있던" 인구가 35일 안에 260 을 넘는가 / 넘은 뒤 무엇을 하는가.
cl = pd.read_csv(DATA / "cohort_climb.csv", dtype=DTYPE, parse_dates=["date"])
cl["elapsed_day"] = (cl["date"] - ANCHOR).dt.days
cl_ok = cl[~cl.error_code.isin(DROP_CODES)].sort_values(["character_name", "date"])
cg = cl_ok.groupby("character_name", observed=True)
n_cl = cg.ngroups
lv0 = cg["level"].first(); lvN = cg["level"].last(); lv_max = cg["level"].max()
_d0dist = {int(k): int(v) for k, v in lv0.value_counts().items()}
print(f"climb 유효 {n_cl}명 (이탈 {cl.character_name.nunique() - n_cl}) | D+0 레벨 {_d0dist}")
print(f"  260 도달(관측 중 최대레벨 ≥ 260) : {(lv_max >= 260).sum():3d}/{n_cl}  ({(lv_max >= 260).mean()*100:.1f}%)   ← approach 대비")
print(f"  261+ 도달                        : {(lv_max >= 261).sum():3d}/{n_cl}  ({(lv_max >= 261).mean()*100:.1f}%)")
print(f"  270+ 도달 (버닝 비욘드 폭주 정황) : {(lv_max >= 270).sum():3d}/{n_cl}")
print(f"  35일간 레벨 전혀 안 오름          : {(lvN - lv0 == 0).sum():3d}/{n_cl}  ({(lvN - lv0 == 0).mean()*100:.0f}%)")

# 260 도달자의 이후 거동 (a 정지 / b 261~269 크롤 / c 270+ 폭주)
reach = lv_max[lv_max >= 260].index
buckets = {"a_260정지": 0, "b_261~269": 0, "c_270+": 0}
day260, er_at260 = [], []
for nm in reach:
    s = cl_ok[cl_ok.character_name == nm]
    hit = s[s.level >= 260]
    day260.append(int(hit.elapsed_day.iloc[0]))
    er_at260.append(float(hit.exp_rate.iloc[0]))
    fin = s.level.iloc[-1]
    buckets["c_270+" if fin >= 270 else "b_261~269" if fin >= 261 else "a_260정지"] += 1
k = len(reach)
print(f"\n[260 도달자 {k}명의 260 이후 거동]  (260 최초 도달일 중앙 D+{int(np.median(day260))})")
for kk, vv in buckets.items():
    print(f"  {kk:10s}: {vv:2d}명 ({vv/k*100:.0f}%)")
print(f"  260 첫 도달 시 exp_rate 중앙 {np.median(er_at260):.1f}% (<5%: {np.mean(np.array(er_at260) < 5)*100:.0f}%) "
      f"→ 레벨 경계에서 즉시 정지 = '260 홀드'")

# 06 climb 궤적 스파게티 (260 도달=색, 나머지=회색)
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
for nm, s in cl_ok.groupby("character_name", observed=True):
    s = s.sort_values("elapsed_day")
    hit260 = s.level.max() >= 260
    ax.plot(s.elapsed_day, s.level,
            color="#C44E52" if hit260 else "0.78",
            lw=1.3 if hit260 else 0.4, alpha=0.9 if hit260 else 0.5,
            zorder=3 if hit260 else 1)
ax.axhline(260, color="k", ls="--", lw=1, label="Lv.260")
ax.set_xlim(0, 34); ax.set_ylim(254, 272)
ax.set_xlabel("앵커 기준 경과일"); ax.set_ylabel("레벨")
ax.set_title(f"climb 코호트 성장 궤적 — 260 도달 {k}명(빨강)은 전원 260에서 정지, 270+ 0명")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
fig.text(0.5, -0.02, CAPTION + " · Lv.255~258 + 앵커 직전 14일 레벨 상승 표본", ha="center", fontsize=8, color="gray")
fig.tight_layout(); fig.savefig(IMG / "06_climb_trajectory.png", bbox_inches="tight"); plt.close(fig)
print("\n저장: images/06_climb_trajectory.png")

climb 유효 300명 (이탈 0) | D+0 레벨 {256: 299, 257: 1}
  260 도달(관측 중 최대레벨 ≥ 260) :  54/300  (18.0%)   ← approach 대비
  261+ 도달                        :   5/300  (1.7%)
  270+ 도달 (버닝 비욘드 폭주 정황) :   0/300
  35일간 레벨 전혀 안 오름          : 208/300  (69%)

[260 도달자 54명의 260 이후 거동]  (260 최초 도달일 중앙 D+23)
  a_260정지   : 49명 (91%)
  b_261~269 :  5명 (9%)
  c_270+    :  0명 (0%)
  260 첫 도달 시 exp_rate 중앙 1.4% (<5%: 72%) → 레벨 경계에서 즉시 정지 = '260 홀드'



저장: images/06_climb_trajectory.png


In [7]:
def ma3(s):
    return s.rolling(3, center=True, min_periods=1).mean()

ret = summary.pivot(index="elapsed_day", columns="cohort", values="retention_rate")
gr  = (clean.groupby(["cohort", "elapsed_day"], observed=True).growth_rate.mean()
       .unstack("cohort"))

# 일간 retention_rate 는 전투력이 8일 간격(D+0/8/16/24/32) 관측이라 stat 날짜에만 튀는 빗살 형태.
# Q2(레벨대별 활동 지속) 비교는 "8일 구간 활동률" — 직전 체크포인트 대비 레벨 OR 전투력 변화 — 으로 본다.
CKPT = [8, 16, 24, 32]
piv = clean.pivot_table(index=["cohort", "character_name"], columns="elapsed_day",
                        values=["level", "combat_power"], observed=True)
win_rows = []
for c in COHORTS:
    for k in CKPT:
        sub = piv.xs(c, level="cohort")
        lvl_ch = sub[("level", k)].ne(sub[("level", k - 8)])
        cp_ch  = sub[("combat_power", k)].ne(sub[("combat_power", k - 8)])
        both_obs = sub[("level", k)].notna() & sub[("level", k - 8)].notna()
        act = ((lvl_ch | cp_ch) & both_obs)
        vals = act[both_obs].astype(float).to_numpy()
        lo, hi = boot_ci(vals)
        win_rows.append({"cohort": c, "ckpt": k, "n": int(both_obs.sum()),
                         "win_retention": float(vals.mean()), "wr_ci_lo": lo, "wr_ci_hi": hi})
winret = pd.DataFrame(win_rows)

print("[8일 구간 활동률 — 직전 체크포인트 대비 레벨 or 전투력 변화]")
print(winret.pivot(index="ckpt", columns="cohort", values="win_retention").round(4))
print("\n[일간 성장률 — elapsed_day 평균 (레벨/일)]")
print(gr.mean().round(4).rename("mean_growth_rate"))
print("\n[참고: 일간 retention_rate 평균 — 빗살 아티팩트 포함]")
print(ret.mean().round(4).rename("mean_retention_daily"))

# 요일 효과 점검 (계절성 기각 근거)
clean["dow"] = clean.date.dt.dayofweek
dow = clean[clean.elapsed_day >= 1].groupby("dow", observed=True).active_t.mean()
print("\n[요일별 평균 활동률 — 월=0 … 일=6]")
print(dow.round(4))
print(f"요일간 표준편차 {dow.std():.4f} | 주말(금~일) {dow.loc[[4,5,6]].mean():.4f} vs 화~목 {dow.loc[[1,2,3]].mean():.4f}")
print("→ 주말이 평일보다 높은 약한 주간 패턴은 있으나 전 코호트 활동률 자체가 <6% 라 트렌드/노이즈 대비 미미. 주 분석에서는 무시(계절성 기각).")

[8일 구간 활동률 — 직전 체크포인트 대비 레벨 or 전투력 변화]
cohort  approach   at260  burnend  past260
ckpt                                      
8         0.0381  0.0583   0.5186   0.1272
16        0.0311  0.0415   0.4949   0.1166
24        0.0242  0.0466   0.4407   0.1166
32        0.0242  0.0284   0.3559   0.0848

[일간 성장률 — elapsed_day 평균 (레벨/일)]
cohort
approach    0.0009
at260       0.0004
past260     0.0002
burnend     0.0031
Name: mean_growth_rate, dtype: Float64

[참고: 일간 retention_rate 평균 — 빗살 아티팩트 포함]
cohort
approach    0.0036
at260       0.0054
burnend     0.0557
past260     0.0133
Name: mean_retention_daily, dtype: float64

[요일별 평균 활동률 — 월=0 … 일=6]
dow
0    0.0194
1    0.0011
2    0.0006
3    0.0005
4    0.0303
5     0.027
6    0.0267
Name: active_t, dtype: Float64
요일간 표준편차 0.0138 | 주말(금~일) 0.0280 vs 화~목 0.0007
→ 주말이 평일보다 높은 약한 주간 패턴은 있으나 전 코호트 활동률 자체가 <6% 라 트렌드/노이즈 대비 미미. 주 분석에서는 무시(계절성 기각).


In [8]:
# 앞 셀의 '활동'은 CP 변화를 이분값(변함/안 변함)으로만 봤다. 여기서는 5개 CP 스냅샷의
# 실제 크기 변화를 본다. D+0 CP=0(무장비 레벨 셸)은 증가율 분모가 되지 못해 제외.
cpv = (clean[clean.combat_power.notna()]
       .pivot_table(index=["cohort", "character_name"], columns="elapsed_day",
                    values="combat_power", observed=True))
lvdelta = (clean.groupby(["cohort", "character_name"], observed=True)["level"]
           .agg(lambda x: (x.dropna().iloc[-1] - x.dropna().iloc[0]) if x.notna().any() else np.nan))

grow_rows = []
for c in COHORTS:
    s = cpv.xs(c, level="cohort").dropna(subset=[0, 32])
    s = s[s[0] > 0]
    d = s[32] - s[0]
    pct = d / s[0] * 100
    ld = lvdelta.xs(c, level="cohort").reindex(s.index).fillna(0)
    real_grow = (d > 0) | (ld > 0)                    # 레벨↑ 또는 전투력↑ (감소·리롤 제외)
    grow_rows.append({
        "cohort": c, "n_CPpos": len(s),
        "n_CP0": int((cpv.xs(c, level="cohort")[0] == 0).sum()),
        "CP증가율중앙%": round(float(pct.median()), 1),
        "CP_p10up": int((pct >= 10).sum()),
        "CP_p25up": int((pct >= 25).sum()),
        "실질성장%": round(float(real_grow.mean()) * 100, 1),
    })
grow = pd.DataFrame(grow_rows)
print("[전투력 성장 규모 — D+0→D+32, D+0 CP>0 캐릭터]")
print("  n_CPpos=D+0 전투력>0 인원, n_CP0=전투력 0(무장비) 제외 인원, CP_p10up/p25up=증가율 10%/25% 이상 인원")
print(grow.to_string(index=False))
print("\n→ '활동'을 레벨↑ 또는 CP↑(감소·장비리롤 제외)로 좁히면 구배: "
      + " / ".join(f"{r.cohort} {r['실질성장%']:.0f}%" for _, r in grow.iterrows()))

# --- burnend 딥다이브 ---
b = clean[clean.cohort == "burnend"].sort_values("date")
b_lv = b.groupby("character_name")["level"].agg(lambda x: x.iloc[-1] - x.iloc[0])
bcp = cpv.xs("burnend", level="cohort").dropna(subset=[0, 32]); bcp = bcp[bcp[0] > 0]
bg = (bcp[32] - bcp[0]) / bcp[0] * 100
print("\n[burnend 딥다이브]")
_lvd = {int(k): int(v) for k, v in b_lv.value_counts().sort_index().items()}
print(f"  레벨업 분포: {_lvd}  (281 고정 대비 +1~2)")
print(f"  CP 증가율: +10%↑ {int((bg>=10).sum())}명 / +25%↑ {int((bg>=25).sum())}명 / +50%↑ {int((bg>=50).sum())}명 (n={len(bg)})")
print(f"  CP 증가율 상위 5: {[round(float(x),1) for x in sorted(bg, reverse=True)[:5]]} %")
print("  8일 구간별 CP 상승/하락/정지 캐릭터:")
for k in CKPT:
    up = int((bcp[k] > bcp[k - 8]).sum()); dn = int((bcp[k] < bcp[k - 8]).sum())
    print(f"    D+{k-8:>2}→{k}: 상승 {up:3d} / 하락 {dn:3d} / 정지 {len(bcp)-up-dn:3d}")

# 05 전투력 성장 분포 (D+0 CP>0, 증가율 클리핑 -50~100%)
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
for c in COHORTS:
    s = cpv.xs(c, level="cohort").dropna(subset=[0, 32]); s = s[s[0] > 0]
    pct = ((s[32] - s[0]) / s[0] * 100).clip(-50, 100)
    ax.hist(pct, bins=np.arange(-50, 101, 5), histtype="step", lw=2,
            color=COLOR[c], label=f"{c} (n={len(s)})")
ax.axvline(0, color="k", lw=0.8, ls=":")
ax.set_xlabel("전투력 증가율 D+0→D+32 (%, ±50~100 클리핑)"); ax.set_ylabel("캐릭터 수")
ax.set_title("전투력 성장 분포 — 260 허들 코호트는 0에 몰림, burnend만 오른쪽 꼬리")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)
fig.text(0.5, -0.02, CAPTION + " · D+0 전투력>0 캐릭터", ha="center", fontsize=8, color="gray")
fig.tight_layout(); fig.savefig(IMG / "05_combat_power_growth.png", bbox_inches="tight"); plt.close(fig)
print("\n저장: images/05_combat_power_growth.png")

[전투력 성장 규모 — D+0→D+32, D+0 CP>0 캐릭터]
  n_CPpos=D+0 전투력>0 인원, n_CP0=전투력 0(무장비) 제외 인원, CP_p10up/p25up=증가율 10%/25% 이상 인원
  cohort  n_CPpos  n_CP0  CP증가율중앙%  CP_p10up  CP_p25up  실질성장%
approach      269     20       0.0         4         1    4.8
   at260      696     76       0.0        17         7    5.5
 past260      258     25       0.0        11         4    9.7
 burnend      284     11       0.0        57        20   31.7

→ '활동'을 레벨↑ 또는 CP↑(감소·장비리롤 제외)로 좁히면 구배: approach 5% / at260 6% / past260 10% / burnend 32%

[burnend 딥다이브]
  레벨업 분포: {0: 266, 1: 27, 2: 2}  (281 고정 대비 +1~2)
  CP 증가율: +10%↑ 57명 / +25%↑ 20명 / +50%↑ 7명 (n=284)
  CP 증가율 상위 5: [227.4, 148.1, 98.0, 92.2, 91.6] %
  8일 구간별 CP 상승/하락/정지 캐릭터:
    D+ 0→8: 상승  61 / 하락  92 / 정지 131
    D+ 8→16: 상승  54 / 하락  92 / 정지 138
    D+16→24: 상승  77 / 하락  53 / 정지 154
    D+24→32: 상승  70 / 하락  35 / 정지 179

저장: images/05_combat_power_growth.png


In [9]:
STALL_DAYS = 3

def stagnation_start(levels):
    lv = pd.Series(levels).to_numpy()
    if pd.isna(lv).all():
        return pd.NA
    run = 1
    for i in range(1, len(lv)):
        if pd.notna(lv[i]) and lv[i] == lv[i - 1]:
            run += 1
            if run >= STALL_DAYS:
                return int(lv[i])
        else:
            run = 1
    return pd.NA        # 관측 내내 성장 지속

pool = []
for (c, nm), sub in clean.groupby(["cohort", "character_name"], observed=True):
    sub = sub.sort_values("date")
    a_lv = sub.level.iloc[0]
    note = None
    if sub.level.isna().all():
        note = "dropout_before_stagnation"
    elif bool(sub.burn_suspect.iloc[0]):
        note = "burning_suspect"
    pool.append({"cohort": c, "character_name": nm,
                 "anchor_level": pd.NA if pd.isna(a_lv) else int(a_lv),
                 "stagnation_start_level": stagnation_start(sub.level),
                 "note": note})
stag = pd.DataFrame(pool)
stag.to_csv(DATA / "stagnation_pooled.csv", index=False, encoding="utf-8-sig")
print("→ data/stagnation_pooled.csv", stag.shape)

s = stag.stagnation_start_level.dropna()
print(f"\n정체 시작 레벨 관측 {len(s)}명 / 전체 {len(stag)}명 (나머지는 관측내 성장지속 or 이탈)")
print("258~260 구간:", int(s.between(258, 260).sum()), f"({s.between(258,260).mean()*100:.0f}%)")
print("280~281 구간:", int(s.between(280, 281).sum()), f"({s.between(280,281).mean()*100:.0f}%)")
print("\n[정체 시작 레벨 분포]")
print(s.value_counts().sort_index())

# --- 유니온 관점 보조증거 : 정지 지점 exp_rate(레벨내 진행도) & 전투력 ---
print("\n[D+0 exp_rate — 레벨 경계에서 멈췄는가]  &  [D+0 전투력 중앙값 — 장비 투자]")
for c in COHORTS:
    d0c = clean[(clean.cohort == c) & (clean.elapsed_day == 0)]
    er = d0c.exp_rate.dropna()
    print(f"  {c:9s}: exp_rate 중앙 {er.median():5.1f}%  (<10% 비율 {er.lt(10).mean()*100:3.0f}%)   "
          f"전투력 중앙 {d0c.combat_power.median():>12,.0f}")

→ data/stagnation_pooled.csv (1643, 5)

정체 시작 레벨 관측 1642명 / 전체 1643명 (나머지는 관측내 성장지속 or 이탈)
258~260 구간: 772 (47%)
280~281 구간: 296 (18%)

[정체 시작 레벨 분포]
stagnation_start_level
250      1
251    289
260    772
262    284
281    296
Name: count, dtype: int64

[D+0 exp_rate — 레벨 경계에서 멈췄는가]  &  [D+0 전투력 중앙값 — 장비 투자]
  approach : exp_rate 중앙   0.1%  (<10% 비율 100%)   전투력 중앙      173,783
  at260    : exp_rate 중앙   6.2%  (<10% 비율 100%)   전투력 중앙    1,407,040
  past260  : exp_rate 중앙   1.9%  (<10% 비율 100%)   전투력 중앙    4,823,341
  burnend  : exp_rate 중앙  43.8%  (<10% 비율   0%)   전투력 중앙   29,544,925


In [10]:
# 01 돌파 곡선
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
for c, lab in [("approach", "approach — 260 도달률"), ("at260", "at260 — 261 이탈률(플래토 탈출)")]:
    d = summary[summary.cohort == c].sort_values("elapsed_day")
    ax.plot(d.elapsed_day, d.breakthrough_rate, marker="o", ms=3, color=COLOR[c], label=lab)
    ax.fill_between(d.elapsed_day, d.bt_ci_lo, d.bt_ci_hi, color=COLOR[c], alpha=0.15)
    last = d.iloc[-1]
    ax.annotate(f"D+34 = {last.breakthrough_rate:.4f}  ({int(round(last.breakthrough_rate*last.n))}/{int(last.n)})",
                xy=(34, last.breakthrough_rate), xytext=(24, 0.14 if c == "approach" else 0.08),
                fontsize=9, color=COLOR[c],
                arrowprops=dict(arrowstyle="->", color=COLOR[c], lw=0.8))
ax.set_ylim(0, 1); ax.set_xlim(0, 34)
ax.set_xlabel("앵커 기준 경과일"); ax.set_ylabel("돌파율 (0~1)")
ax.set_title("260 돌파율 — 성수기 이벤트 기간 (앵커 6/18)")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)
fig.text(0.5, -0.02, CAPTION, ha="center", fontsize=8, color="gray")
fig.tight_layout(); fig.savefig(IMG / "01_breakthrough_curve.png", bbox_inches="tight"); plt.close(fig)

# 02 레벨대별 활동 지속률 (8일 구간)
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
for c in COHORTS:
    d = winret[winret.cohort == c].sort_values("ckpt")
    lw = 2.8 if c == "at260" else 1.8
    ax.plot(d.ckpt, d.win_retention, marker="o", ms=5, color=COLOR[c], lw=lw, label=c)
    ax.fill_between(d.ckpt, d.wr_ci_lo, d.wr_ci_hi, color=COLOR[c], alpha=0.15)
ax.set_ylim(0, 0.6); ax.set_xticks(CKPT); ax.set_xlim(6, 34)
ax.set_xlabel("앵커 기준 경과일 (8일 체크포인트)"); ax.set_ylabel("구간 활동률 (직전 체크포인트 대비)")
ax.set_title("레벨대별 활동 지속률 — 8일 구간 (레벨 or 전투력 변화, at260 강조)")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)
fig.text(0.5, -0.02, CAPTION + " · 8일 구간 (전투력 관측 간격)", ha="center", fontsize=8, color="gray")
fig.tight_layout(); fig.savefig(IMG / "02_retention_by-level-tier.png", bbox_inches="tight"); plt.close(fig)

# 03 정체 시작 레벨 지도
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
bins = np.arange(250, 301, 1)
stacks = [stag.loc[stag.cohort == c, "stagnation_start_level"].dropna().astype(int) for c in COHORTS]
ax.hist(stacks, bins=bins, stacked=True, color=[COLOR[c] for c in COHORTS], label=COHORTS)
ax.axvspan(258, 260, color="orange", alpha=0.18, label="258–260 (난이도·버닝종료·유니온)")
ax.axvspan(280, 281, color="red", alpha=0.15, label="280–281 (버닝 BEYOND 종료)")
ax.set_xlabel("정체 시작 레벨"); ax.set_ylabel("캐릭터 수")
ax.set_title("정체 시작 레벨 분포 (4코호트 풀링, STALL_DAYS=3)")
ax.legend(loc="upper right"); ax.grid(alpha=0.3)
fig.text(0.5, -0.02, CAPTION + " · bin=1", ha="center", fontsize=8, color="gray")
fig.tight_layout(); fig.savefig(IMG / "03_stagnation_map.png", bbox_inches="tight"); plt.close(fig)

# 04 성장 궤적 스파게티 (approach + at260) — 버닝추정 0명이라 '레벨 변동 캐릭터'를 강조색으로
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
sp = clean[clean.cohort.isin(["approach", "at260"])]
n_flat = {"approach": 0, "at260": 0}
movers = 0
for (c, nm), sub in sp.groupby(["cohort", "character_name"], observed=True):
    sub = sub.sort_values("elapsed_day")
    moved = bool(sub.level.dropna().nunique() > 1)
    if moved:
        ax.plot(sub.elapsed_day, sub.level, color="#C44E52", lw=1.6, alpha=0.95, zorder=3)
        movers += 1
    else:
        ax.plot(sub.elapsed_day, sub.level, color="0.75", lw=0.6, alpha=0.5, zorder=1)
        if pd.notna(sub.level.iloc[0]):
            n_flat[c] += 1
ax.axhline(260, color="k", ls="--", lw=1, label="Lv.260")
ax.set_xlim(0, 34); ax.set_ylim(248, 285)
ax.set_xlabel("앵커 기준 경과일"); ax.set_ylabel("레벨")
ax.set_title("성장 궤적 — approach+at260 (레벨 변동=빨강, 정지=회색)")
ax.text(0.5, 251.4, f"Lv.251 정지 {n_flat['approach']}명", fontsize=9, color="0.35")
ax.text(0.5, 260.6, f"Lv.260 정지 {n_flat['at260']}명", fontsize=9, color="0.35")
ax.text(34, 283, f"레벨 변동 {movers}명", fontsize=9, color="#C44E52", ha="right")
ax.legend(loc="center left"); ax.grid(alpha=0.3)
fig.text(0.5, -0.02, CAPTION, ha="center", fontsize=8, color="gray")
fig.tight_layout(); fig.savefig(IMG / "04_growth_trajectory.png", bbox_inches="tight"); plt.close(fig)

print("저장:", sorted(p.name for p in IMG.glob("*.png")))

저장: ['01_breakthrough_curve.png', '02_retention_by-level-tier.png', '03_stagnation_map.png', '04_growth_trajectory.png', '05_combat_power_growth.png', '06_climb_trajectory.png']


In [11]:
# 인사이트용 핵심 수치 재출력 (아래 markdown 셀 근거)
last = summary[summary.elapsed_day == 34]
print("approach 260 도달률 D+34 :", round(float(last[last.cohort=='approach'].breakthrough_rate.iloc[0]), 4))
print("at260 261 이탈률 D+34     :", round(float(last[last.cohort=='at260'].breakthrough_rate.iloc[0]), 4))
print("리텐션 평균:", ret.mean().round(3).to_dict())
print("활동 base-rate:", base["활동"].round(3).to_dict())
print("정체 258~260 집중:", int(stag.stagnation_start_level.dropna().between(258,260).sum()))

approach 260 도달률 D+34 : 0.0035
at260 261 이탈률 D+34     : 0.0039
리텐션 평균: {'approach': 0.004, 'at260': 0.005, 'burnend': 0.056, 'past260': 0.013}
활동 base-rate: {'approach': 0.09, 'at260': 0.12, 'past260': 0.218, 'burnend': 0.639}
정체 258~260 집중: 772


## 인사이트 (관찰 Fact / 원인 Why / 행동 Action)

### 인사이트 1 — "260 이후 방치"의 상당 부분은 실패가 아니라 설계된 종착점
- **관찰**: `approach`(251)·`at260`(260)·`past260`(262) 세 코호트 모두 35일간 활동 캐릭터 비율 9~22%,
  레벨 변동 캐릭터는 각 1명 이하. 정지 캐릭터의 D+0 `exp_rate` 는 `approach` 중앙 0.1%(전원<1%),
  `at260` 전원 <10% — **레벨을 찍은 그 순간 멈췄다**. 반면 `burnend`(281)는 `exp_rate` 중앙 43.8%,
  활동 64%로 질적으로 다름.
- **원인(가설)**: 메이플 유니온(리전)은 부캐 레벨 합이 본캐 스펙을 올리는 구조라, 버닝으로 특정 레벨
  (250·260 근방 브레이크포인트)까지 올린 뒤 **의도적으로 파킹**하는 플레이가 일반적. 260을 넘기면
  경험치는 폭증하고 유니온 한계효용은 줄어 넘길 유인이 없음. 260 플래토의 대량 캐릭터는 "막힌 유저"가
  아니라 **완성된 유니온 인프라**이고, 소유자는 다른 캐릭터에서 활동 중일 가능성.
  (반례: 일부는 실제 탈퇴 계정 — 둘은 스냅샷으로 분리 불가.)
- **행동**: "260 돌파율"을 KPI 로 쓰면 안 됨. 유니온 부캐의 생애주기(생성→버닝→파킹)를 별도 세그먼트로
  보고, 정체를 "이탈 위험"이 아니라 "목적 달성"으로 재분류하는 지표 재설계.

### 인사이트 2 — 활동성을 가르는 건 레벨대가 아니라 캐릭터 성장도(전투력)
- **관찰**: D+0 전투력 중앙값이 코호트 순서대로 `approach` 17만 → `at260` 140만 → `past260` 482만
  → `burnend` 2,954만 (약 170배). 활동 비율도 같은 순서로 9→12→22→64%. 코호트 **내부**에서도 활동
  캐릭터의 전투력이 정지 캐릭터의 1.5~2배.
- **원인(가설)**: 낮은 전투력 = 장비 투자 없이 레벨만 올린 캐릭터 = 유니온용 부캐일 개연성. 전투력이
  높은(=본캐로 육성된) 캐릭터일수록 이벤트 기간에도 계속 성장. "260 허들"은 난이도 장벽이라기보다
  **본캐/부캐 성격의 프록시**로 작동.
- **행동**: 레벨 기반 세그먼트(계획문서 세부질문 2)의 실효성이 낮음. 전투력·과금·유니온 레벨 등
  **성장 투자 지표**로 세그먼트를 재정의해야 리텐션 타겟팅이 유효.

### 인사이트 3 — 여름 성장 이벤트는 이미 파킹된 캐릭터를 되살리지 못한다
- **관찰**: 성수기(이벤트 진행 중) 앵커임에도 `approach` 260 도달률 D+34 ≈ 0.003, `at260` 261 이탈률
  ≈ 0.005. 버닝추정군은 `approach`·`at260` 통틀어 사실상 0명 → "버닝추정 vs 비버닝 정착률 비교"
  (계획문서 세부질문 1)는 수행 불가.
- **원인(가설)**: 하이퍼 버닝은 계정당 1캐릭터(신규 or 기존 200~258)만 지정 가능. 이미 260에 파킹된
  캐릭터는 지정 대상이 아니라, 이벤트 효과가 이 코호트에 도달할 경로 자체가 없음.
- **행동**: 이벤트 ROI 를 "기존 정체 캐릭터 재활성화"로 기대하면 안 됨. 이벤트의 실제 타겟은
  신규·저레벨 유니온 부캐 생성이며, 그 지표(신규 캐릭터 생성 수, 250 도달 시간)로 성과를 측정.

> 관찰은 모두 넥슨 API 레벨/전투력 스냅샷 기반 대리지표. 유니온 소속·계정 연결·실접속 로그는
> API 로 확인 불가하며, 위 원인은 정황 근거에 기반한 가설이다 (계획문서 §12).